# Lean-42 — L'indépendance arithmétique, certifiée : PA ne tranche pas une machine de 196 états

**Navigation** : [Index](README.md) | [<< Lean-41 Capstone MIMO](Lean-41-Capstone-MIMO-Python.ipynb) | [English version](Lean-42-Independance-Arithmetique-BusyBeaver-Lean_en.ipynb)

**Lake** : [`busybeaver_lean/`](busybeaver_lean/README.md) — consommateur épinglé du paquet `PA196` de [jamesyc/BusyBeaver](https://github.com/jamesyc/BusyBeaver).

**Source** : papier *PA196* (MIT), archivé au gisement `BusyBeaver/PA196/paper/pa196.pdf`. Issue de série : #27477, jalon 2.


## Plan

1. **Le lake consommateur** : où vit le paquet amont, à quel commit, avec quel toolchain.
2. **Le pin** : relire le manifeste et vérifier que la révision tient.
3. **Le build** : construire le témoin, lire ce que `lake` a réellement fait.
4. **L'audit d'axiomes** : `#print axioms` sur les quatre théorèmes centraux, et le verdict mécanique.
5. **Les nombres du papier** : états, lignes de table, pas — confrontés à la table de transition du paquet amont.
6. **Lecture** : ce que « indépendant de PA » veut dire, et ce que cela ne dit pas.
7. **Exercices** : trois, à écrire dans les cellules stub.


## 1. Le lake consommateur

Le lake `busybeaver_lean` ne définit **rien** : il épingle le paquet `PA196` du dépôt amont et importe ses théorèmes. La clause du `lakefile.lean` est directe :

```lean
require «pa196» from git
  "https://github.com/jamesyc/BusyBeaver.git" @ "8419afeb…" / "PA196"
```

La forme `/ "PA196"` est la syntaxe Lake du **sous-répertoire** : Lake clone le dépôt et charge le paquet tel qu'il est publié, sans fork ni copie. C'est la forme la plus stricte de la doctrine `CONSUMER_PINNED` du dépôt : aucun module amont n'est vendu, adapté ou recopié.

Le toolchain du lake est **v4.35.0**, celui du paquet amont, et non celui du lake voisin `formal_logic_lean` (4.33.1). Ce n'est pas un oubli : le corpus `PA196` est volumineux et son `vendor/Foundation` est lui-même en 4.35.0. Le ramener à 4.33.1 demanderait de porter tout le corpus ; un lake par toolchain est la forme prévue par Lake, et les deux lakes vivent côte à côte.

La cellule suivante localise le lake, calcule le chemin WSL, et compare les empreintes SHA-256 des fichiers de build entre le dépôt et le lake effectivement utilisé : un build réussi dans ce lake vaut pour le dépôt **seulement** si les quatre empreintes concordent.


In [1]:
import hashlib
import os
import pathlib
import subprocess

NB_DIR = pathlib.Path.cwd()
REPO_LAKE = NB_DIR / "busybeaver_lean"
PIN_ATTENDU = "8419afeb6513c2ec5a14d4cbd0d2ade43d5553b0"


def wslpath(p):
    # Chemin Windows -> chemin WSL, calcule par WSL lui-meme.
    r = subprocess.run(["wsl", "-e", "wslpath", "-a", str(p).replace(chr(92), "/")],
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    return r.stdout.strip()


def wsl_sh(commande):
    # Execute une commande shell dans WSL et rend sa sortie standard.
    r = subprocess.run(["wsl", "-e", "bash", "-lc", commande],
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    return r.stdout


LAKE_OVERRIDE = os.environ.get("BUSYBEAVER_LAKE_WSL")
LAKE_DIR = LAKE_OVERRIDE or wslpath(REPO_LAKE)
LEAN_EXEC = wslpath(NB_DIR.parents[2] / "scripts" / "lean" / "lean_exec.py")


def lean_exec(commande, timeout=900):
    # Lance `commande` dans le lake via l'organe borne ; rend (code, sortie, journal).
    r = subprocess.run(
        ["wsl", "-e", "bash", "-lc",
         f"cd {LAKE_DIR} && python3 {LEAN_EXEC} run --timeout {timeout} -- {commande}"],
        capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=timeout + 120)
    journal = [l for l in r.stderr.splitlines() if l.startswith("[lean_exec]")]
    return r.returncode, r.stdout, journal


FICHIERS_BUILD = ["lean-toolchain", "lakefile.lean", "lake-manifest.json", "BusyBeaver.lean"]

# Tout est accumule puis imprime en UNE fois : un seul flush stdout, donc une
# seule sortie commitee -- le jumeau EN ne peut pas decouper autrement.
lignes = ["lake utilise : " + ("copie (BUSYBEAVER_LAKE_WSL)" if LAKE_OVERRIDE
                               else "busybeaver_lean du depot")]
cote_wsl = wsl_sh(f"cd {LAKE_DIR} && sha256sum " + " ".join(FICHIERS_BUILD)).splitlines()
empreintes_wsl = {l.split()[1]: l.split()[0] for l in cote_wsl if l.strip()}
identiques = 0
for nom in FICHIERS_BUILD:
    h_depot = hashlib.sha256((REPO_LAKE / nom).read_bytes()).hexdigest()
    h_lake = empreintes_wsl.get(nom, "absent")
    identiques += h_depot == h_lake
    lignes.append(f"  {nom:22s} depot {h_depot[:12]}  lake {h_lake[:12]}  "
                  f"{'identique' if h_depot == h_lake else 'DIFFERENT'}")
lignes.append(f"{identiques}/{len(FICHIERS_BUILD)} fichiers identiques"
              + ("  (comparaison triviale : meme dossier)" if not LAKE_OVERRIDE else ""))

# Le manifeste peut annoncer un pin ; la tete du clone reellement construit le
# prouve. Les deux sont lus : le manifeste a la cellule suivante, la tete ici.
tete = wsl_sh(f"git -C {LAKE_DIR}/.lake/packages/pa196 rev-parse HEAD").strip()
lignes.append("tete du clone amont : " + tete)
lignes.append("pin tenu (clone)    : " + str(tete == PIN_ATTENDU))
lignes.append("toolchain           : " + (REPO_LAKE / "lean-toolchain").read_text().strip())
print("\n".join(lignes))

lake utilise : busybeaver_lean du depot
  lean-toolchain         depot ead308ab4fc7  lake ead308ab4fc7  identique
  lakefile.lean          depot e7e789afae0a  lake e7e789afae0a  identique
  lake-manifest.json     depot 0c2ef4f790b7  lake 0c2ef4f790b7  identique
  BusyBeaver.lean        depot d5b36af89c18  lake d5b36af89c18  identique
4/4 fichiers identiques  (comparaison triviale : meme dossier)
tete du clone amont : 8419afeb6513c2ec5a14d4cbd0d2ade43d5553b0
pin tenu (clone)    : True
toolchain           : leanprover/lean4:v4.35.0


### Lecture de la mise en place

Quatre lignes, une par fichier de build : chacune affiche l'empreinte côté dépôt et l'empreinte côté lake. Toutes doivent porter `identique` — sinon le lake construit est un autre objet que celui du dépôt (copie oubliée, état intermédiaire). En mode copie (`BUSYBEAVER_LAKE_WSL`) la comparaison est réelle ; sur le lake du dépôt elle est trivialement vraie, et la cellule le dit plutôt que de le laisser croire.

Deux lignes ferment le point aveugle du manifeste : `git rev-parse HEAD` sur le clone amont rend la révision **réellement extraite**, et `pin tenu (clone)` doit valoir `True`. Un manifeste peut annoncer un pin que le clone n'a pas ; la tête du clone ne le peut pas. Le toolchain affiché est celui que `lake` utilisera — `v4.35.0`.


## 2. Le pin, relu dans le manifeste

`lake update` a résolu la clause `from git` en une révision inscrite dans `lake-manifest.json`. Ce fichier est le contrat : toute construction ultérieure part de ces révisions, pas de la branche amont.

La cellule lit le manifeste et cherche le paquet `pa196`. On attend le commit `8419afeb…` — celui sur lequel l'amont publie son paquet, son audit et son papier. Un manifeste qui porterait une autre révision serait une dérive silencieuse : le build réussirait, mais sur un autre contenu.


In [2]:
# Le manifeste Lake fige le commit exact du paquet : c'est la preuve du pin.
import json

manifeste = json.loads((REPO_LAKE / "lake-manifest.json").read_text(encoding="utf-8"))
# PA196 est la dependance directe ; les autres paquets viennent de ses transitives.
paquet = next(p for p in manifeste["packages"] if p["name"] == "pa196")
print("paquet      :", paquet["name"])
print("rev         :", paquet["rev"])
print("url         :", paquet["url"])
print("pin tenu    :", paquet["rev"] == PIN_ATTENDU)
print("sous-dir lu :", paquet.get("subDir"))
print("dependances :", ", ".join(sorted(p["name"] for p in manifeste["packages"]
                                       if p["name"] != "pa196")))

paquet      : pa196
rev         : 8419afeb6513c2ec5a14d4cbd0d2ade43d5553b0
url         : https://github.com/jamesyc/BusyBeaver.git
pin tenu    : True
sous-dir lu : PA196
dependances : Cli, Foundation, LeanSearchClient, Qq, aesop, batteries, importGraph, mathlib, plausible, proofwidgets


### Lecture du pin

Quatre valeurs à confronter : `rev` doit égaler le pin attendu (`pin tenu : True`), `url` doit désigner `jamesyc/BusyBeaver`, `sous-dir lu` doit valoir `PA196`, et la liste des dépendances doit être celle que le paquet amont déclare. Ces dépendances — Mathlib, le `Foundation` vendorisé, puis `aesop`, `batteries`, `Qq`, `Cli`, `plausible`, `proofwidgets`, `import-graph`, `LeanSearchClient` — sont résolues par le paquet amont lui-même : ce lake ne les re-déclare pas, donc il ne peut pas les faire diverger.


## 3. Le build

La cellule lance `lake build` dans le lake, par l'organe `scripts/lean/lean_exec.py` : cap machine-wide, confinement de l'arbre de processus, budget mesuré. C'est le même organe que les autres carnets Lean de la série ; aucun `lake` n'est lancé à la main.

Sur un lake neuf, deux commandes précèdent : `lake update` (résolution du pin et écriture du manifeste) puis `lake exe cache get` (oléans Mathlib de la révision amont). Sans le cache, Mathlib se compilerait depuis les sources — plusieurs heures pour un contenu que l'amont publie déjà. Le carnet les tient pour acquises : son objet est le build du témoin, pas la préparation du lake.

Un premier build paie ensuite la compilation du paquet amont — `ProofSources` et son `vendor/Foundation` — et les suivants sont incrémentaux. Le temps affiché n'est donc pas une propriété du résultat : c'est l'état du cache.


In [3]:
# Build du temoin : incremental une fois les oleans en place.
code_retour, sortie, journal = lean_exec("lake build", timeout=5400)
utiles = [l for l in sortie.splitlines()
          if not l.startswith(("info:", "trace", "  ", "\u2139"))]
print("\n".join(utiles[-8:]))
# Le temps de build varie d'un run a l'autre : normalise, sinon le jumeau EN
# ne peut pas porter la meme sortie (garde twin-parity).
import re
print("\n".join(re.sub(r"duration=[0-9.]+s", "duration=mesuree", l) for l in journal))
print("code de retour :", code_retour)


 Quot.sound]
 Classical.choice,
 Quot.sound]
 Classical.choice,
 Quot.sound]
 Classical.choice,
 Quot.sound]
Build completed successfully (3195 jobs).
[lean_exec] ok exit=0 duration=mesuree orphans=0
[lean_exec] child_exit_code: 0
code de retour : 0


### Lecture du build

Trois choses se lisent ici. Le **code de retour** doit être nul — un `lake build` qui échoue rend un code non nul et des lignes d'erreur. Le **journal de l'organe** porte la mesure d'admission et la postcondition « zéro descendant orphelin ». Enfin, la sortie utile nomme les cibles construites : `BusyBeaver`, et derrière elle les bibliothèques du paquet amont qu'elle importe (`PA196`, `M6`, `M6PA`, `PA196Bridge`).

Un build incrémental peut n'afficher aucune tâche : c'est le signe que les `.olean` sont à jour, pas que rien n'a été vérifié. Le fichier `BusyBeaver.lean` et ses imports ont alors été élaborés lors d'un build antérieur.

Le module amont `PA196.Audit` est un audit **gardé** : ses `#print axioms` sont encadrés par des `#guard_msgs` qui figent l'ensemble attendu. Construire `PA196` exécute donc ces gardes — un axiome supplémentaire, une preuve laissée vide ou un changement de confiance native y font échouer la compilation, avant même les reimpressions de ce carnet.


## 4. L'audit d'axiomes

Lean permet de demander, pour chaque théorème, la liste des axiomes dont sa preuve dépend. C'est la mesure qui distingue une preuve complète d'une preuve qui s'appuie sur `sorry` (`sorryAx`) ou sur l'évaluation native (`Lean.ofReduceBool`, via `native_decide`).

Le module `BusyBeaver.lean` écrit quatre `#print axioms` : `PA196.machine_nonhalting`, `PA196.Arithmetic.nonhalting_unprovable`, `PA196.Arithmetic.halting_unprovable`, `PA196.Arithmetic.two_sided_independence`. La cellule ré-élabore le module avec `lake env lean` — les imports sont résolus par le lake — et lit les lignes d'axiomes.

Le verdict est ensuite mécanique, pas déclaratif : la cellule suivante compare l'ensemble cité à la liste blanche `propext`, `Classical.choice`, `Quot.sound`, et à la liste interdite `sorryAx`, `Lean.ofReduceBool`, `Lean.trustCompiler`.


In [4]:
# Audit d'axiomes : `lake env lean` re-elabore le module et imprime chaque
# ligne `#print axioms` ecrite dans BusyBeaver.lean.
code_retour, sortie, journal = lean_exec("lake env lean BusyBeaver.lean", timeout=1800)
lignes_axiomes = [l.strip() for l in sortie.splitlines() if "axioms" in l.lower()]
print("\n".join(lignes_axiomes))
print("code de retour :", code_retour)


'PA196.machine_nonhalting' depends on axioms: [propext, Classical.choice, Quot.sound]
'PA196.Arithmetic.nonhalting_unprovable' depends on axioms: [propext, Classical.choice, Quot.sound]
'PA196.Arithmetic.halting_unprovable' depends on axioms: [propext, Classical.choice, Quot.sound]
'PA196.Arithmetic.two_sided_independence' depends on axioms: [propext, Classical.choice, Quot.sound]
code de retour : 0


In [5]:
AUTORISES = {"propext", "Classical.choice", "Quot.sound"}
INTERDITS = {"sorryAx", "Lean.ofReduceBool", "Lean.trustCompiler"}


def axiomes_cites(lignes):
    # Extrait l'ensemble des axiomes cites par les lignes `#print axioms`.
    trouves = set()
    for ligne in lignes:
        if "depends on axioms:" in ligne:
            liste = ligne.split("depends on axioms:")[-1].strip().strip("[]")
            trouves |= {a.strip() for a in liste.split(",") if a.strip()}
        # « does not depend on any axioms » : ensemble vide, rien a ajouter.
    return trouves


cites = axiomes_cites(lignes_axiomes)
print("axiomes cites :", sorted(cites) if cites else "(aucun)")
print("hors liste blanche :", sorted(cites - AUTORISES) or "(aucun)")
print("interdits presents :", sorted(cites & INTERDITS) or "(aucun)")
print("audit concluant :", cites <= AUTORISES and not (cites & INTERDITS))

axiomes cites : ['Classical.choice', 'Quot.sound', 'propext']
hors liste blanche : (aucun)
interdits presents : (aucun)
audit concluant : True


### Lecture de l'audit

Chaque théorème doit rendre une ligne `depends on axioms: …` dont le contenu est un sous-ensemble de la liste blanche, ou `does not depend on any axioms`. Les trois axiomes autorisés sont ceux du noyau de Lean et de Mathlib : `propext` (propositional extensionality), `Classical.choice` (choix classique), `Quot.sound` (quotients). Aucun n'est un trou de preuve.

`audit concluant : True` est le seul résultat acceptable. Un `sorryAx` signalerait une preuve laissée vide chez l'amont ; un `Lean.ofReduceBool` signalerait un appel à `native_decide`, dont la confiance repose sur le compilateur plutôt que sur le noyau.

Ce verdict mécanique double un garde que l'amont porte lui-même : `PA196.Audit` encadre ses `#print axioms` de `#guard_msgs` qui figent l'ensemble attendu, si bien qu'un axiome supplémentaire fait échouer la compilation du paquet — donc notre build, avant nos propres lignes. Le carnet mesure la même chose dans son propre journal, et à la même révision.


## 5. Les nombres du papier

Le résultat est un théorème, mais son énoncé cite des nombres : 196 états, une table de transition complète, 58 154 pas pour atteindre le cadre du programme de registres, et une réduction de 13 états sur la construction publiée précédemment (209 états).

La cellule lit la table de transition du paquet consommé (`data/pa196.tm`) et confronte ses nombres — `2 × 196 = 392`, `209 − 196 = 13` — puis calcule un ordre de grandeur. Elle ne re-prouve rien : la preuve est celle de l'amont, consommée par le lake. Confronter un énoncé à sa donnée est une hygiène de lecture, pas une revendication de résultat.


In [6]:
# Les nombres du papier, verifies sur la donnee amont : la table de transition
# `data/pa196.tm` est lue ligne a ligne, pas recopiee depuis le texte.
chemin_table = f"{LAKE_DIR}/.lake/packages/pa196/PA196/data/pa196.tm"
lignes_table = [l for l in wsl_sh(f"cat {chemin_table}").splitlines() if l.strip()]
etats_lus = sorted({int(l.split()[0]) for l in lignes_table})
symboles_lus = sorted({l.split()[1] for l in lignes_table})
ETATS, ROWS, PAS_VERS_CADRE, ETATS_PRECEDENT = 196, 392, 58_154, 209

print("lignes de table lues      :", len(lignes_table), "| papier :", ROWS,
      "| concordance :", len(lignes_table) == ROWS)
print("etats distincts lus       :", len(etats_lus), "| papier :", ETATS,
      "| concordance :", etats_lus == list(range(ETATS)))
print("symboles lus              :", symboles_lus,
      "| alphabet binaire :", symboles_lus == ["0", "1"])
print("lignes attendues 2*196    :", 2 * ETATS, "| lues :", len(lignes_table),
      "| concordance :", 2 * ETATS == len(lignes_table))
print("reduction d'etats 209-196 :", ETATS_PRECEDENT - ETATS, "| papier : 13")
print("pas par etat (ordre de grandeur) :", round(PAS_VERS_CADRE / ETATS, 1))

lignes de table lues      : 392 | papier : 392 | concordance : True
etats distincts lus       : 196 | papier : 196 | concordance : True
symboles lus              : ['0', '1'] | alphabet binaire : True
lignes attendues 2*196    : 392 | lues : 392 | concordance : True
reduction d'etats 209-196 : 13 | papier : 13
pas par etat (ordre de grandeur) : 296.7


### Lecture des nombres

Les concordances valent `True` : la table lue dans `data/pa196.tm` rend 196 états distincts exactement `0…195`, un alphabet binaire, et `2 × 196 = 392`. La table n'est pas recopiée du texte du papier : elle est lue dans le paquet consommé, à la révision épinglée. L'ordre de grandeur du coût de la machine (environ trois cents pas par état) est un chiffre de lecture, sans statut de théorème.

Ce qui est certifié n'est pas ce quotient : c'est l'indépendance de l'énoncé d'arrêt vis-à-vis de PA, et la non-arrêt dans le modèle standard.


## 6. Lecture : ce que « indépendant de PA » veut dire

Trois énoncés se répondent dans l'audit, et il vaut la peine de les séparer :

| Théorème | Ce qu'il établit |
|---|---|
| `machine_nonhalting` | la machine ne s'arrête pas, dans le modèle standard |
| `halting_unprovable` | PA ne prouve pas que la machine s'arrête |
| `nonhalting_unprovable` | PA ne prouve pas qu'elle ne s'arrête pas |
| `two_sided_independence` | les deux précédents, réunis : l'énoncé d'arrêt est indépendant de PA |

Le pivot est le second théorème d'incomplétude de Gödel : la machine cherche une réfutation dans PA, et PA prouve que sa non-arrêt implique sa propre cohérence. Si PA prouvait la non-arrêt, elle prouverait sa cohérence — exclu si elle est cohérente. La soundness dans le modèle standard exclut l'autre côté.

Ce que cela **ne dit pas** : ni la valeur de BB(196), ni une borne sur le nombre d'états nécessaires. Le résultat est un témoin d'indépendance, pas un record de la fonction du castor affairé. C'est exactement le versant formel qui manquait à Complexity-07b : le carnet y décrit les grands nombres, celui-ci montre une preuve machine-vérifiée d'un énoncé d'indépendance.

Le lake voisin `formal_logic_lean` (Lean-03b, Lean-34b) porte la logique de prouvabilité et FairBot ; `busybeaver_lean` porte l'indépendance arithmétique. Les deux consomment des paquets épinglés, à des toolchains différents, et ne se dupliquent pas.


### Exercice 1 — Le prédicat d'audit

In [7]:
# Exercice 1 : ecrire le predicat d'audit qui refuserait un axiome interdit.
def audit_concluant(lignes):
    # TODO etudiant : renvoyer True si tous les axiomes cites sont dans AUTORISES.
    resultat = None  # TODO etudiant
    return resultat


print("Exercice a completer : audit_concluant(lignes_axiomes)")


Exercice a completer : audit_concluant(lignes_axiomes)


### Exercice 2 — États et lignes de table

In [8]:
# Exercice 2 : generaliser la relation etats -> lignes d'une machine binaire.
def nombre_de_lignes(etats):
    # TODO etudiant : une machine a `etats` etats sur un alphabet binaire a
    # deux lignes par etat (une par symbole lu).
    resultat = None  # TODO etudiant
    return resultat


print("Exercice a completer : nombre_de_lignes(196)")


Exercice a completer : nombre_de_lignes(196)


### Exercice 3 — Indépendance n'est pas valeur de BB

In [9]:
# Exercice 3 : distinguer l'independance d'un enonce de la valeur BB(196).
def classe_du_resultat(non_halting_prouve, independant_de_pa):
    # TODO etudiant : renvoyer "valeur BB" si le resultat donne une borne sur
    # BB(196), "independance" s'il ne porte que sur la prouvabilite dans PA.
    resultat = None  # TODO etudiant
    return resultat


print("Exercice a completer : classe_du_resultat(True, True)")


Exercice a completer : classe_du_resultat(True, True)


## 7. Conclusion

- Le paquet `PA196` est consommé **tel quel**, au commit épinglé, par la clause `from git … / "PA196"` : aucun fork, aucune copie, aucune adaptation.
- Le build passe sous l'organe borné, et l'audit d'axiomes ne cite que `propext`, `Classical.choice`, `Quot.sound` — l'audit **gardé** de l'amont (`#guard_msgs`) compile dans le même build : il ne pouvait pas en être autrement sans faire échouer la compilation.
- Le résultat est un **témoin d'indépendance** : PA ne tranche pas l'arrêt de cette machine de 196 états, et celle-ci ne s'arrête pas dans le modèle standard.
- Les nombres du papier sont confrontés à la donnée amont : `data/pa196.tm` rend 196 états exactement `0…195` et un alphabet binaire ; les 58 154 pas et la réduction de 13 états restent des valeurs de lecture, sans re-preuve.

### Références

- Papier *PA196* (MIT) : `G:\Mon Drive\MyIA\IA\Bibliographie IA\BusyBeaver\PA196\paper\pa196.pdf`
- Dépôt : [jamesyc/BusyBeaver](https://github.com/jamesyc/BusyBeaver), commit `8419afeb6513c2ec5a14d4cbd0d2ade43d5553b0`
- Série : [Complexity-07b — Le tour des grands nombres](../../InformationCalcul/Complexity/Complexity-07b-LeTourDesGrandsNombres-Python.ipynb)
- Issue : #27477 (jalon 2)